# Performance Metrics and Rolling Windows

**Chapter 8: Feature Engineering for Soccer Predictions**

## Learning Objectives

- Understand why feature engineering is crucial for model performance
- Learn to calculate rolling window statistics for recent form
- Create separate home and away performance metrics
- Build shot quality features beyond simple counts
- Engineer passing and possession features
- Understand data leakage and how to avoid it

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Feature Engineering Toolkit Ready!")

## Why Feature Engineering Matters: The Secret Sauce

Before we dive into code, let's understand why feature engineering is so crucial.

> **"Garbage in, garbage out."**

You can have the most sophisticated neural network in the world, but if you feed it poorly constructed features, it's going to struggle. It's like having Lionel Messi on your team but never passing him the ball in dangerous positions—you're wasting his talent.

### Good Features Do Four Things:

**1. Capture Domain Knowledge**

As soccer analysts, we know that a team's recent form matters more than their performance from six months ago. We know that home advantage is real. We know that beating a top team is more impressive than beating a bottom-dweller. By encoding this knowledge into features, we give our models a head start.

**2. Reduce Complexity**

Raw event data can be overwhelming—thousands of passes, shots, and tackles. A single match in the StatsBomb dataset might have 3,000+ individual events. By aggregating and transforming this data into meaningful features, we create a more digestible representation of what happened.

**3. Improve Model Performance**

This is the bottom line. A simple logistic regression with great features will often outperform a deep neural network with mediocre features. Feature engineering is where you get the biggest bang for your buck. In many Kaggle competitions and real-world applications, the winning teams don't have the fanciest models—they have the best features.

**4. Increase Interpretability**

When your model predicts that Netherlands will beat Japan, you want to know **why**. With well-engineered features, you can say: "The model predicted a Netherlands win because they had a higher Colley rating (0.68 vs 0.54), better recent form (2.3 goals per game vs 1.8), and a higher xG per shot (0.12 vs 0.09)."

## Generate Simulated Match Data

For this notebook, we'll create simulated match data to demonstrate feature engineering techniques.

In [ ]:
# Generate simulated season data
np.random.seed(42)

# Teams
teams = ['Arsenal', 'Chelsea', 'Liverpool', 'Man City', 'Man United', 'Tottenham',
         'Leicester', 'West Ham', 'Everton', 'Aston Villa', 'Newcastle', 'Brighton',
         'Crystal Palace', 'Wolves', 'Southampton', 'Burnley', 'Leeds', 'Watford',
         'Norwich', 'Brentford']

# Generate matches (each team plays each other twice)
matches = []
match_id = 1
start_date = datetime(2023, 8, 12)

for round_num in range(2):  # Home and away
    for i, home_team in enumerate(teams):
        for j, away_team in enumerate(teams):
            if i != j:
                # Generate match data
                match_date = start_date + timedelta(days=match_id * 4)
                
                # Simulate scores (Poisson-ish distribution)
                home_score = np.random.poisson(1.5)
                away_score = np.random.poisson(1.0)
                
                # Simulate match statistics
                home_shots = np.random.poisson(12)
                away_shots = np.random.poisson(10)
                
                home_shots_on_target = int(home_shots * np.random.uniform(0.3, 0.5))
                away_shots_on_target = int(away_shots * np.random.uniform(0.3, 0.5))
                
                home_xg = home_shots * np.random.uniform(0.08, 0.15)
                away_xg = away_shots * np.random.uniform(0.08, 0.15)
                
                home_possession = np.random.uniform(40, 60)
                away_possession = 100 - home_possession
                
                home_passes = int(home_possession * np.random.uniform(8, 12))
                away_passes = int(away_possession * np.random.uniform(8, 12))
                
                home_pass_accuracy = np.random.uniform(75, 90)
                away_pass_accuracy = np.random.uniform(75, 90)
                
                matches.append({
                    'match_id': match_id,
                    'match_date': match_date,
                    'home_team': home_team,
                    'away_team': away_team,
                    'home_score': home_score,
                    'away_score': away_score,
                    'home_shots': home_shots,
                    'away_shots': away_shots,
                    'home_shots_on_target': home_shots_on_target,
                    'away_shots_on_target': away_shots_on_target,
                    'home_xg': home_xg,
                    'away_xg': away_xg,
                    'home_possession': home_possession,
                    'away_possession': away_possession,
                    'home_passes': home_passes,
                    'away_passes': away_passes,
                    'home_pass_accuracy': home_pass_accuracy,
                    'away_pass_accuracy': away_pass_accuracy
                })
                
                match_id += 1

matches_df = pd.DataFrame(matches)
matches_df = matches_df.sort_values('match_date').reset_index(drop=True)

print(f"Generated {len(matches_df)} matches")
print(f"Date range: {matches_df['match_date'].min()} to {matches_df['match_date'].max()}")
print(f"\nSample matches:")
matches_df.head()

---

# Performance Metrics: Quantifying the Basics

Performance metrics are the bread and butter of soccer analytics. They capture a team's historical form and tendencies in numbers. These are the stats you see on TV broadcasts and in post-match reports, but we're going to be more sophisticated about how we calculate and use them.

## Goals Scored and Conceded: The Rolling Window

Goals are the ultimate currency in soccer. A team that has been scoring freely in their last few games is likely confident and dangerous. Their attackers are in form, their midfielders are creating chances, and the whole team is playing with swagger.

But here's the thing: we don't want just the total goals from the entire season. A team that scored 30 goals in their first 10 matches but only 5 in their last 10 matches is not the same team anymore. We want to know their **recent form**.

### The Rolling Window Concept

A rolling window is like a **moving spotlight** that illuminates only the most recent matches. As new matches are played, the window slides forward, always showing you the last N matches. It's the data science equivalent of asking "What have you done for me lately?"

In [ ]:
# Create a long-form DataFrame of goals per team per match
all_teams_scores = pd.concat([
    matches_df[['match_date', 'home_team', 'home_score']].rename(
        columns={'home_team': 'team', 'home_score': 'goals'}),
    matches_df[['match_date', 'away_team', 'away_score']].rename(
        columns={'away_team': 'team', 'away_score': 'goals'})
]).sort_values(by=['team', 'match_date'])

print("Long-form data structure:")
print(all_teams_scores.head(10))
print(f"\nTotal rows: {len(all_teams_scores)}")

### Calculating Rolling Averages

We'll calculate rolling averages over 3, 5, and 10-match windows to capture short, medium, and long-term form.

In [ ]:
# Calculate 3-match rolling average
all_teams_scores['goals_rolling_avg_3'] = all_teams_scores.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)  # ⚠️ shift(1) prevents data leakage!
)

# Also calculate 5-match and 10-match windows
all_teams_scores['goals_rolling_avg_5'] = all_teams_scores.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

all_teams_scores['goals_rolling_avg_10'] = all_teams_scores.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean().shift(1)
)

print("Rolling averages calculated!")
print("\nExample for one team:")
print(all_teams_scores[all_teams_scores['team'] == 'Arsenal'].head(15))

### Understanding the Code

**1. Long-form reshape**: We reshape the data so each row is one team's performance in one match. This makes it easy to calculate team-specific features.

**2. `rolling()` function**: Creates a moving window over the data. Here, we're averaging the last 3 matches. Think of it as asking "What's this team's average goals per game in their last 3 outings?"

**3. `shift(1)` is crucial!** ⚠️ This ensures we're only using **past matches** to predict the current one. Without this, we'd be "cheating" by including the current match's result in our feature, which we wouldn't have at prediction time. This is called **data leakage**, and it's one of the most common mistakes in machine learning.

**4. Multiple window sizes**: Different windows capture different aspects of form:
   - **3-match window**: Very sensitive to recent changes
   - **5-match window**: Balances recent and medium-term form
   - **10-match window**: Reveals longer-term trends

In [ ]:
# Visualize rolling averages for one team
team_example = 'Liverpool'
team_data = all_teams_scores[all_teams_scores['team'] == team_example].reset_index(drop=True)

plt.figure(figsize=(14, 6))
plt.plot(team_data.index, team_data['goals'], 'o-', label='Actual Goals', alpha=0.6, markersize=4)
plt.plot(team_data.index, team_data['goals_rolling_avg_3'], label='3-Match Avg', linewidth=2)
plt.plot(team_data.index, team_data['goals_rolling_avg_5'], label='5-Match Avg', linewidth=2)
plt.plot(team_data.index, team_data['goals_rolling_avg_10'], label='10-Match Avg', linewidth=2)
plt.xlabel('Match Number', fontsize=12)
plt.ylabel('Goals', fontsize=12)
plt.title(f'{team_example} - Goals and Rolling Averages', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Notice how the 3-match average reacts quickly to changes,")
print(f"while the 10-match average is smoother and more stable.")

### Rolling Standard Deviation: Measuring Consistency

In [ ]:
# Calculate rolling standard deviation to measure consistency
all_teams_scores['goals_rolling_std_5'] = all_teams_scores.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=5, min_periods=1).std().shift(1)
)

print("Rolling standard deviation calculated!")
print("\nTeams with high std are volatile (unpredictable)")
print("Teams with low std are consistent\n")

# Find most and least consistent teams
team_consistency = all_teams_scores.groupby('team')['goals_rolling_std_5'].mean().sort_values()

print("Most Consistent Teams (low std):")
print(team_consistency.head(5))
print("\nMost Volatile Teams (high std):")
print(team_consistency.tail(5))

---

## Home vs. Away: The Tale of Two Teams

One of the most well-documented phenomena in soccer is **home advantage**. Teams simply perform better when playing in their own stadium, in front of their own fans, with familiar surroundings and no travel fatigue.

The home advantage in soccer is worth roughly **0.3-0.5 goals per game** on average, though it varies by team and league.

### The Nuance

Not all teams benefit equally from home advantage. Some teams are fortress-like at home but struggle on the road (think Borussia Dortmund with their Yellow Wall). Other teams are more consistent regardless of venue.

When predicting Manchester United vs. Manchester City, it's more relevant to look at:
- Man United's **home** form
- Man City's **away** form

Not their overall form!

In [ ]:
# Calculate separate home and away rolling averages

# Home performance
home_data = matches_df[['match_date', 'home_team', 'home_score']].rename(
    columns={'home_team': 'team', 'home_score': 'goals'}
).copy()
home_data['venue'] = 'home'

home_data = home_data.sort_values(['team', 'match_date'])
home_data['goals_rolling_avg_home_3'] = home_data.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
)

# Away performance
away_data = matches_df[['match_date', 'away_team', 'away_score']].rename(
    columns={'away_team': 'team', 'away_score': 'goals'}
).copy()
away_data['venue'] = 'away'

away_data = away_data.sort_values(['team', 'match_date'])
away_data['goals_rolling_avg_away_3'] = away_data.groupby('team')['goals'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
)

print("Separate home/away rolling averages calculated!")
print("\nHome performance (sample):")
print(home_data[home_data['team'] == 'Arsenal'].head())
print("\nAway performance (sample):")
print(away_data[away_data['team'] == 'Arsenal'].head())

In [ ]:
# Compare home vs away performance for all teams
home_avg = home_data.groupby('team')['goals'].mean()
away_avg = away_data.groupby('team')['goals'].mean()

comparison = pd.DataFrame({
    'home_avg': home_avg,
    'away_avg': away_avg,
    'home_advantage': home_avg - away_avg
}).sort_values('home_advantage', ascending=False)

print("Home Advantage Analysis:")
print(comparison)

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
teams_sorted = comparison.sort_values('home_advantage').index
y_pos = np.arange(len(teams_sorted))

ax.barh(y_pos, comparison.loc[teams_sorted, 'home_advantage'], 
        color=['green' if x > 0 else 'red' for x in comparison.loc[teams_sorted, 'home_advantage']])
ax.set_yticks(y_pos)
ax.set_yticklabels(teams_sorted)
ax.set_xlabel('Home Advantage (goals per game)', fontsize=12)
ax.set_title('Home Advantage by Team', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---

## Shot Quality: Beyond Just Counting

Not all shots are created equal. A shot from 30 yards out with three defenders in the way is not the same as a one-on-one with the goalkeeper from 6 yards.

This is where **Expected Goals (xG)** comes in. xG quantifies the quality of a shot based on factors like:
- Distance from goal
- Angle to goal
- Body part used (foot, head)
- Type of assist
- Number of defenders

A shot with an xG of 0.8 has an 80% chance of being scored, while a shot with xG of 0.05 has only a 5% chance.

In [ ]:
# Calculate xG-based features

# xG per shot (shot quality)
matches_df['home_xg_per_shot'] = matches_df['home_xg'] / matches_df['home_shots'].replace(0, 1)
matches_df['away_xg_per_shot'] = matches_df['away_xg'] / matches_df['away_shots'].replace(0, 1)

# Shot conversion rate (goals / shots)
matches_df['home_shot_conversion'] = matches_df['home_score'] / matches_df['home_shots'].replace(0, 1)
matches_df['away_shot_conversion'] = matches_df['away_score'] / matches_df['away_shots'].replace(0, 1)

# xG overperformance (actual goals - xG)
matches_df['home_xg_diff'] = matches_df['home_score'] - matches_df['home_xg']
matches_df['away_xg_diff'] = matches_df['away_score'] - matches_df['away_xg']

print("Shot quality features calculated!")
print("\nSample:")
print(matches_df[['home_team', 'away_team', 'home_shots', 'home_xg', 
                  'home_xg_per_shot', 'home_score', 'home_xg_diff']].head(10))

In [ ]:
# Visualize xG vs actual goals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Home teams
axes[0].scatter(matches_df['home_xg'], matches_df['home_score'], alpha=0.5)
axes[0].plot([0, matches_df['home_xg'].max()], [0, matches_df['home_xg'].max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Expected Goals (xG)', fontsize=12)
axes[0].set_ylabel('Actual Goals', fontsize=12)
axes[0].set_title('Home Teams: xG vs Actual Goals', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Away teams
axes[1].scatter(matches_df['away_xg'], matches_df['away_score'], alpha=0.5, color='orange')
axes[1].plot([0, matches_df['away_xg'].max()], [0, matches_df['away_xg'].max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Expected Goals (xG)', fontsize=12)
axes[1].set_ylabel('Actual Goals', fontsize=12)
axes[1].set_title('Away Teams: xG vs Actual Goals', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Points above the line = overperforming xG (clinical finishing or luck)")
print("Points below the line = underperforming xG (poor finishing or unlucky)")

---

## Passing and Possession: Style and Control

Passing and possession statistics tell us about a team's style of play and their ability to control the game.

### Key Metrics:
- **Possession %**: How much of the ball they have
- **Pass accuracy**: Quality of their passing
- **Passes per game**: Volume of passing (possession-based vs direct)

Some teams (like Barcelona or Man City) dominate possession and make hundreds of passes. Others (like Burnley or Atletico Madrid) are more direct and efficient.

In [ ]:
# Calculate passing features

# Successful passes (total passes * accuracy)
matches_df['home_successful_passes'] = matches_df['home_passes'] * (matches_df['home_pass_accuracy'] / 100)
matches_df['away_successful_passes'] = matches_df['away_passes'] * (matches_df['away_pass_accuracy'] / 100)

# Create long-form for rolling calculations
passing_data = pd.concat([
    matches_df[['match_date', 'home_team', 'home_possession', 'home_pass_accuracy']].rename(
        columns={'home_team': 'team', 'home_possession': 'possession', 'home_pass_accuracy': 'pass_accuracy'}),
    matches_df[['match_date', 'away_team', 'away_possession', 'away_pass_accuracy']].rename(
        columns={'away_team': 'team', 'away_possession': 'possession', 'away_pass_accuracy': 'pass_accuracy'})
]).sort_values(['team', 'match_date'])

# Rolling averages for possession and passing
passing_data['possession_rolling_avg_5'] = passing_data.groupby('team')['possession'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

passing_data['pass_accuracy_rolling_avg_5'] = passing_data.groupby('team')['pass_accuracy'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

print("Passing features calculated!")
print("\nSample:")
print(passing_data[passing_data['team'] == 'Man City'].head(10))

In [ ]:
# Analyze team playing styles
team_style = passing_data.groupby('team').agg({
    'possession': 'mean',
    'pass_accuracy': 'mean'
}).round(2)

team_style.columns = ['avg_possession', 'avg_pass_accuracy']
team_style = team_style.sort_values('avg_possession', ascending=False)

print("Team Playing Styles:")
print(team_style)

# Visualize
plt.figure(figsize=(10, 8))
plt.scatter(team_style['avg_possession'], team_style['avg_pass_accuracy'], s=100, alpha=0.6)

for team, row in team_style.iterrows():
    plt.annotate(team, (row['avg_possession'], row['avg_pass_accuracy']), 
                fontsize=8, alpha=0.7, xytext=(3, 3), textcoords='offset points')

plt.xlabel('Average Possession %', fontsize=12)
plt.ylabel('Average Pass Accuracy %', fontsize=12)
plt.title('Team Playing Styles: Possession vs Passing Accuracy', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop-right = Possession-based teams (high possession, high accuracy)")
print("Bottom-left = Direct teams (low possession, lower accuracy)")

## Summary

In this notebook, you learned:

✅ **Why feature engineering matters** - the secret sauce of machine learning

✅ **Rolling windows** for capturing recent form (3, 5, 10-match averages)

✅ **Data leakage prevention** with `shift(1)`

✅ **Home vs away splits** to capture venue-specific performance

✅ **Shot quality metrics** using xG

✅ **Passing and possession features** to understand playing style

✅ **Consistency metrics** using rolling standard deviation

### Key Takeaways

**1. Recent form matters more than season-long averages**
- Use rolling windows to capture "What have you done for me lately?"

**2. Context is crucial**
- Separate home and away performance
- Not all teams benefit equally from home advantage

**3. Quality over quantity**
- xG per shot > total shots
- Pass accuracy matters as much as possession

**4. Avoid data leakage!**
- Always use `shift(1)` in rolling calculations
- Only use information available at prediction time

**5. Multiple perspectives**
- Create features at different time scales (3, 5, 10 matches)
- Let the model decide which matters most

## Practice Exercises

1. **Goals conceded**: Create rolling averages for goals conceded (defensive form).

2. **Form streaks**: Create a feature that counts consecutive wins/losses.

3. **Shots on target ratio**: Calculate rolling averages for shots on target / total shots.

4. **xG efficiency**: Create a rolling average of (goals / xG) to measure finishing quality.

5. **Possession dominance**: Calculate the difference between home and away possession.

6. **Weighted rolling average**: Give more weight to recent matches (exponential moving average).

7. **Head-to-head**: Create features for historical performance against specific opponents.

8. **Time decay**: Implement a feature where older matches have less influence.